In [84]:
# Exceptions

class ImageEncoderError(Exception):
    """Custom exception for errors related to image encoding."""
    def __init__(self, message: str):
        super().__init__(message)

class MessageBuilderError(Exception):
    """Custom exception for errors related to message building."""
    def __init__(self, message: str):
        super().__init__(message)

class APIClientError(Exception):
    """Custom exception for errors related to OpenRouter client operations."""
    def __init__(self, message: str):
        super().__init__(message)

In [85]:
# Message API

import numpy as np
import cv2
import os
import base64
import json

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Dict, List, Optional, Union, Any
from enum import Enum, auto
from pathlib import Path
from openai import OpenAI

class MessageRole(Enum):
    SYSTEM = "system"
    DEVELOPER = "developer"
    USER = "user"
    ASSISTANT = "assistant"

class ContentType(Enum):
    TEXT = auto()
    IMAGE = auto()

@dataclass(frozen=True, slots=True)
class MessageContent:
    type: ContentType
    data: str

@dataclass(frozen=True, slots=True)
class Message:
    role : MessageRole
    content : Union[str, List[MessageContent]]

class ImageEncoder:
    @staticmethod
    def encode_image(image: Union[Path, np.ndarray]) -> str:
        def resize_image(img: np.ndarray, scale: float = 0.5) -> np.ndarray:
            height, width = img.shape[:2]
            new_size = (int(width * scale), int(height * scale))
            return cv2.resize(img, new_size, interpolation=cv2.INTER_AREA)

        if isinstance(image, Path):
            try:
                if not image.exists():
                    raise ImageEncoderError(f"Image file {image} does not exist.")

                if image.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
                    raise ImageEncoderError(f"Unsupported image format: {image.suffix}. Supported formats are .jpg, .jpeg, and .png.")

                img = cv2.imread(str(image))
                if img is None:
                    raise ImageEncoderError(f"Failed to read image file {image} using OpenCV.")

                resized_img = resize_image(img)

                _, buffer = cv2.imencode(image.suffix.lower(), resized_img)
                b64_image = base64.b64encode(buffer).decode('utf-8')
                mime_type = 'jpeg' if image.suffix.lower() in ['.jpg', '.jpeg'] else 'png'
                b64_image_str = f"data:image/{mime_type};base64,{b64_image}"

                # with open(image, "rb") as img_file:
                #     b64_image = base64.b64encode(img_file.read()).decode('utf-8')
                #     b64_image_str = f"data:image/{image.suffix[1:]};base64,{b64_image}"

            except IOError as e:
                raise ImageEncoderError(f"Error reading image file {image}: {e}")

        elif isinstance(image, np.ndarray):
            if image.ndim != 3 or image.shape[2] not in [3, 4]:
                raise ImageEncoderError("Invalid image array shape. Expected a 3D array with 3 (RGB) or 4 (RGBA) channels.")

            resized_img = resize_image(image)

            _, buffer = cv2.imencode('.png', image)
            b64_image = base64.b64encode(buffer).decode('utf-8')
            b64_image_str = f"data:image/png;base64,{b64_image}"

        else:
            raise TypeError("Image must be a numpy array or a Path object pointing to an image file.")

        return b64_image_str

class ContentBuilder:
    @staticmethod
    def create_text_content(text : str) -> MessageContent:
        return MessageContent(type=ContentType.TEXT, data=text)

    @staticmethod
    def create_image_content(image: Union[Path, np.ndarray]) -> MessageContent:
        b64_image_str = ImageEncoder.encode_image(image)
        return MessageContent(type=ContentType.IMAGE, data=b64_image_str)

class MessageBuilder:
    def __init__(self, role : MessageRole):
        self._role = role
        self._content: List[MessageContent] = []

    def add_text_content(self, text: str):
        self._content.append(ContentBuilder.create_text_content(text))

    def add_image_content(self, image: Union[Path, np.ndarray]):
        self._content.append(ContentBuilder.create_image_content(image))

    def build_message(self) -> Message:
        message = None

        if not self._role:
            raise MessageBuilderError("Message role cannot be empty")

        if not self._content:
            raise MessageBuilderError("Message content cannot be empty")

        if len(self._content) == 1:
            # If there's only one content item, return it directly
            message = Message(role=self._role, content=self._content[0].data)
        else:
            # If there are multiple content items, return them as a list
            message = Message(role=self._role, content=self._content)

        return message

In [86]:
# API Clients

class APIClient(ABC):
    @abstractmethod
    def send_message(self, messages: List[Message], **kwargs) -> str:
        """Send a message to the API and return the response."""
        pass

    @abstractmethod
    def _format_messages(self, messages: List[Message]) -> List[Dict[str, str]]:
        """Format messages for the API request."""
        pass

    @abstractmethod
    def _format_content(self, content: MessageContent) -> Dict[str, str]:
        """Format content for the API request."""
        pass

class OpenAIClient(APIClient):
    def __init__(self, model_name: str):
        self.api_key = os.getenv("OPENAI_API_KEY")
        if not self.api_key:
            raise APIClientError("OPENAI_API_KEY environment variable is not set.")

        self.client = OpenAI(api_key=self.api_key)
        self.model_name = model_name

    def _format_messages(self, messages: List[Message]) -> List[Dict[str, str]]:
        formatted_messages = []
        for message in messages:
            formatted_message = {
                "role": message.role.value,
                "content": message.content if isinstance(message.content, str) else [self._format_content(content) for content in message.content]
            }
            formatted_messages.append(formatted_message)
        return formatted_messages

    def _format_content(self, content: MessageContent) -> Dict[str, str]:
        if content.type == ContentType.TEXT:
            return {"type": "input_text", "text": content.data}
        elif content.type == ContentType.IMAGE:
            return {"type": "input_image", "image_url": content.data}
        else:
            raise APIClientError(f"Unsupported content type: {content.type}")

    def send_message(self, messages: List[Message], **kwargs) -> str:
        if not messages:
            raise APIClientError("Messages cannot be empty.")

        formatted_messages = self._format_messages(messages)
        response = self.client.responses.create(
            model="gpt-4.1-mini-2025-04-14",
            input=formatted_messages,
            temperature=0.0,
            max_output_tokens=500,
        )

        return response.output_text

class OpenRouterClient(APIClient):
    BASE_URL = "https://openrouter.ai/api/v1"
    def __init__(self, model_name: str):
        self.api_key = os.getenv("OPENROUTER_API_KEY")
        if not self.api_key:
            raise APIClientError("OPENROUTER_API_KEY environment variable is not set.")

        self.client = OpenAI(api_key=self.api_key, base_url=OpenRouterClient.BASE_URL)
        self.model_name = model_name

    def _format_messages(self, messages: List[Message]) -> List[Dict[str, str]]:
        formatted_messages = []
        for message in messages:
            formatted_message = {
                "role": message.role.value,
                "content": message.content if isinstance(message.content, str) else [self._format_content(content) for content in message.content]
            }
            formatted_messages.append(formatted_message)
        return formatted_messages


    def _format_content(self, content: MessageContent) -> Dict[str, str]:
        if content.type == ContentType.TEXT:
            return {"type": "text", "text": content.data}
        elif content.type == ContentType.IMAGE:
            return {
                "type": "image_url",
                "image_url": {
                    "url" : content.data
                }
            }
        else:
            raise APIClientError(f"Unsupported content type: {content.type}")

    def send_message(self, messages: List[Message], **kwargs) -> str:
        if not messages:
            raise APIClientError("Messages cannot be empty.")

        formatted_messages = self._format_messages(messages)

        response = self.client.chat.completions.create(
            model=self.model_name,
            extra_body={},
            messages=formatted_messages,
            temperature=0.0,
            max_completion_tokens=250,
        )

        return response.choices[0].message.content

In [87]:
# Model Registry and Instantiator
import base64
import os
import pathlib
from dataclasses import dataclass, asdict
from enum import Enum, auto
from typing import Any, Iterable, List, Mapping, Protocol, Sequence

class ModelCatalogue:
    _catalogue : Mapping[str, Sequence[str]] = {
        "openai": ["openai/gpt-4.1-2025-04-14"],
        "qwen": ["qwen/qwen2.5-vl-72b-instruct:free"],
        "google": ["google/gemini-2.5-flash"],
        "anthropic": ["anthropic/claude-3-7-sonnet-20250219"],
    }

    @classmethod
    def validate(cls, model_name: str):
        provider = model_name.split("/", 1)[0]
        if provider not in cls._catalogue:
            raise ValueError(
                f"Unknown provider '{provider}'. Valid providers: {list(cls._catalogue)}"
            )
        if model_name not in cls._catalogue[provider]:
            raise ValueError(
                f"Unknown model '{model_name}' for provider '{provider}'. "
                f"Choices: {cls._catalogue[provider]}"
            )

    @classmethod
    def providers(cls) -> Sequence[str]:
        return tuple(cls._catalogue)

    @classmethod
    def models(cls, provider: str) -> Sequence[str]:
        return tuple(cls._catalogue[provider])

class VLMAgent:
    def __init__(self, model_name: str, **kwargs: Any):
        # ModelCatalogue.validate(model_name)

        self.model_name = model_name
        # self.client = OpenRouterClient(self.model_name)
        self.client = OpenAIClient(self.model_name)

        self.temperature = kwargs.get("temperature", 0.1)
        self.max_output_tokens = kwargs.get("max_output_tokens", 500)

    def create_user_message(self, text : str = None, image : Union[Path, np.ndarray] = None) -> Message:
        builder = MessageBuilder(MessageRole.USER)

        if text:
            builder.add_text_content(text)

        if image:
            builder.add_image_content(image)

        return builder.build_message()

    def create_system_message(self, text: str) -> Message:
        builder = MessageBuilder(MessageRole.DEVELOPER)
        # builder = MessageBuilder(MessageRole.SYSTEM)
        builder.add_text_content(text)
        return builder.build_message()

    def send_message(self, messages: List[Message]) -> str:
        if not messages:
            raise ValueError("Messages cannot be empty.")

        return self.client.send_message(messages, temperature=self.temperature, max_output_tokens=self.max_output_tokens)

In [ ]:
class SceneAnalyzer(VLMAgent):
    def __init__(
            self,
            model_name: str = "qwen/qwen2.5-vl-72b-instruct:free",
            **kwargs):
        super().__init__(model_name, kwargs=kwargs)

    def interpret_scene(self, scene_context_path: str, image_path: Path):
#         system_instruction = f"""
# You are an expert autonomous driving assistant. You will be given RGB images of the Bird's Eye View (BEV) and
# front-view with respect to the ego vehicle. Alongside this information, you will be provided with a textual
# representation of the scene metadata. Given this information, provide a concise, natural language summary of
# the current driving context.

# Also, identify relevant actors in the scene based on their pose and proximity to
# the ego vehicle. Note that leading vehicles are more important than trailing vehicles in the ego lane, whereas
# both are important in adjacent lanes. In the textual summary, lane adjacency is provided as (DIRECTION - n) where
# DIRECTION is either "Left" or "Right" and n is the lane number relative to the ego vehicle's lane.

# Your response should be structured as follows:

# road_description: Description of the road network (junction, highway, local road, etc.).
# traffic_description: Description of the traffic conditions in the scene.
# static_objects_and_obstacles_description: Description of static objects and obstacles in the scene (if present).
# ego_vehicle_description: Description of the state of the ego vehicle.
# key_actors: Description of nearby actors with their IDs, poses, and states that are relevant to the driving context.
# reasoning: Brief description justifying your choice of key actors.
# """
        system_instruction = f"""
You are a component of an expert autonomous driving assistant, responsible for the perception stack of the pipeline.
You will be given RGB images of the Bird's Eye View (BEV) and front-view with respect to the ego vehicle. Alongside
this information, you will be provided with a textual representation of the scene metadata. Given this information,
provide a concise, natural language summary of the current driving context.

#ENVIRONMENT SETUP#
A 2D BEV coordinate system is used for decision-making and all measurements use the metric system. The setup is as follows:
    1. All positions are given as 2D coordinates in the x-y plane in metres. The x-axis is oriented positive UP and the y-axis is oriented
       positive RIGHT
    2. All orientations are equivalent to the yaw and are given in radians
    3. All distance measurements are given in metres
    4. All speed measurements are given in metres/second

#IMAGE INPUT DETAILS#
In both the front-view and BEV RGB images, all NPC objects are enclosed by their ground-truth bounding boxes and labeled with
their corresponding actor IDs. In the image input, the front-view image is stacked on top of the BEV image. Attend carefully
to critical objects and actors in the front-view image and use the BEV image for additional surrounding context to inform
your decisions.

#TEXT INPUT DETAILS#
The text scene representation contains data split into 3 categories: traffic, ego, and agent context. Traffic context includes
information such as next traffic light distance and state or next stop sign distance if applicable, alongside the road speed limit.
Ego context includes current speed, orientation, 2D [x, y] BEV position, and upcoming lane changes in the
global ego route. Finally, agent context includes NPC vehicle data grouped into 3 traffic types: ongoing, oncoming, and cross. If a traffic
type is not present, it is not included in the text input. Within these groups, the vehicles are further categorized based on their lanes with
respect to the current ego lane and whether they are leading or trailing the ego vehicle. Lane adjacency is provided as (DIRECTION - n) where DIRECTION
is either "Left" or "Right" and n is the lane number relative to the ego vehicle's lane.

##NPC VEHICLE INPUT DETAILS##
The NPC vehicle data includes their actor ID which is connected to their bounding box and label in the image input. They also include their relative
position, orientation, and distance with respect to the ego vehicle with the measurements expressed in the 2D coordinate system from above. Finally, their
absolute speed is also provided.

#PERCEPTION TASK#
You are responsible for two perception tasks:
1. Condense the image and text input into a concise natural language summary of the current driving context. This will be passed on to the prediction
component of the system to understand the scene and predict how the environment will evolve in the next few seconds

2. Identify the relevant actors that are important for the ego vehicle's decision-making process. Reason step by step to determine which actors need
to be considered. Focus on their pose and proximity to the ego vehicle. Also, inspect the BEV and front-view bounding boxes' alignment with key traffic elements
(e.g. lane markings, intersections, sidewalks, etc.) to assess upcoming or ongoing maneuvers. Also note that leading vehicles are more important than trailing vehicles in the ego lane, whereas
both are important in directly adjacent lanes.

Your response should be structured as follows:

road_description: Description of the road network (junction, highway, local road, etc.).
traffic_description: Description of the traffic conditions in the scene.
static_objects_and_obstacles_description: Description of static objects and obstacles in the scene (if present).
ego_vehicle_description: Description of the state of the ego vehicle.
key_actors: Description of nearby actors with their IDs, poses, and states that are relevant to the driving context.
reasoning: Step-by-step reasoning justifying your choice of key actors
        """

        with open(scene_context_path, 'r') as file:
            scene_context = file.read()

        system_message = self.create_system_message(text=system_instruction)
        user_message = self.create_user_message(text=scene_context, image=image_path)

        messages = [system_message, user_message]

        response = self.send_message(messages)

        return response

    def predict_intentions(self, scene_context_path: str, image_path: Path, key_actor_description: str):
#         system_instruction = f"""
# You are an expert autonomous driving assistant. You will be given RGB images of the Bird's Eye View (BEV) and
# front-view with respect to the ego vehicle. Alongside this information, you will be provided with a textual
# representation of the scene metadata and a corresponding natural language summary. This includes information
# about key actors that may be relevant to the ego vehicle's immediate decision-making process.

# Given this information, predict the likely intentions of the key actors in the scene considering ego vehicle's
# remaining route which is shown as a green line in the image. Note that NPC actors are simplistic and do not
# account for the ego vehicle's state when making decisions. Thus, opt for NPC actor predictions that allow
# the ego to prioritize its own safety and make conservative decisions.

# For all actors, consider their current pose, speed, and proximity to the ego vehicle. For vehicles, their set of
# possible actions are: "follow_lane", "change_lane_left", "change_lane_right", "turn_left", "turn_right",
# "stop". For pedestrians, their actions are: "cross_street", "wait", and "walk".

# Your response should be structured as follows:

# key_actor_id: ID of the key actor.
# key_actor_type: Type of the key actor (e.g., vehicle, pedestrian).
# key_actor_intention: Predicted intention of the key actor based on the scene context.
# key_actor_reasoning: Brief description justifying the predicted intention of the key actor.
#         """
        system_instruction = f"""
You are a component of an expert autonomous driving assistant, responsible for the prediction stack of the pipeline.
You will be given RGB images of the Bird's Eye View (BEV) and front-view with respect to the ego vehicle. Alongside
this information, you will be provided with a corresponding natural language summary and list of key actors generated
from the upstream perception stack. Given this information, predict the likely intentions of the key actors in the scene.

#ENVIRONMENT SETUP#
A 2D BEV coordinate system is used for decision-making and all measurements use the metric system. The setup is as follows:
    1. All positions are given as 2D coordinates in the x-y plane in metres. The x-axis is oriented positive UP and the y-axis is oriented
       positive RIGHT
    2. All orientations are equivalent to the yaw and are given in radians
    3. All distance measurements are given in metres
    4. All speed measurements are given in metres/second

#IMAGE INPUT DETAILS#
In both the front-view and BEV RGB images, all NPC objects are enclosed by their ground-truth bounding boxes and labeled with
their corresponding actor IDs. In the image input, the front-view image is stacked on top of the BEV image. Attend carefully
to critical objects and actors in the front-view image and use the BEV image for additional surrounding context to inform
your decisions. The green line represents the global route the ego vehicle is expected to follow.

#TEXT INPUT DETAILS#
The natural language summary describes the scene as follows:

road_description: Description of the road network (junction, highway, local road, etc.).
traffic_description: Description of the traffic conditions in the scene.
static_objects_and_obstacles_description: Description of static objects and obstacles in the scene (if present).
ego_vehicle_description: Description of the state of the ego vehicle.
key_actors: Description of nearby actors with their IDs, poses, and states that are relevant to the driving context.
reasoning: Step-by-step reasoning justifying the choice of key actors

#PREDICTION TASK#
You are responsible for the following prediction task:
1. Predict the likely intentions of the key actors in the scene considering their pose, speed, and proximity to the ego vehicle. Use the BEV bounding box alignment and
lane markings to determine if the actor is partially overlapping adjacent lanes or offset from lane center. Also inspect bounding box tilt and lateral shift
in the front-view image to assess ongoing maneuvers. These spatial cues are stronger signals of imminent lane changes than orientation angles alone.
NPC actors execute maneuvers based solely on their own local lane position and heading. Therefore, you must prioritize pose and lateral offset
from lane center when identifying potential lane changes. Slight orientation angles can still indicate significant intent when combined with
lateral displacement. Use the visual bounding box information in the BEV and front-view images to detect such deviations. Assume risk-aware predictions that
prioritize the ego vehicle's safety by anticipating the most plausible maneuver with the greatest potential impact on the ego, even if the maneuver
is not the most statistically likely. NPC actors are not reactive to the ego vehicle, so you must plan as though they will not yield or
correct mistakes. When visual or behavioral cues suggest even a small likelihood of an intrusive or disruptive maneuver (e.g., lane change, sudden stop, intersection turn),
predict that maneuver to ensure safe planning. Reason step by step to explain why each actor is expected to execute the predicted maneuver.

##NPC ACTOR AVAILABLE ACTIONS##
All NPC actors only execute a set of discrete actions that are listed below:

###VEHICLES###
1. `follow_lane`
2. `change_lane_left`
3. `change_lane_right`
4. `turn_left`
5. `turn_right`
6. `stop`

###PEDESTRIANS###
1. `cross_street`
2. `wait`
3. `walk`

Your response should be structured as follows:

key_actor_id: ID of the key actor.
key_actor_type: Type of the key actor (e.g., vehicle, pedestrian).
key_actor_intention: Predicted intention of the key actor based on the scene context.
key_actor_reasoning: Step-by-step reasoning justifying the predicted intention of the key actor
        """

        with open(scene_context_path, 'r') as file:
            scene_context = file.read()

        system_message = self.create_system_message(text=system_instruction)

        # separator_text = "\n\nKey Actor Description:\n"
        # scene_context_with_key_actor = scene_context + separator_text + key_actor_description
        scene_context_with_key_actor = key_actor_description
        user_message = self.create_user_message(text=scene_context_with_key_actor, image=image_path)

        messages = [system_message, user_message]

        response = self.send_message(messages)

        return response

    def plan_ego_actions(self, key_actor_intentions: str, image_path: Path):
#         system_instruction = f"""
# You are an expert autonomous driving assistant. You will be given RGB images of the Bird's Eye View (BEV) and
# front-view with respect to the ego vehicle. Alongside this information, you will be provided with a textual
# summary of likely intentions of key actors in the scene.

# Given this information, generate a sequence of high-level driving actions for the ego vehicle over a 3s planning
# horizon. The plan should prioritize the ego vehicle's safety and conservative decision-making, taking into account the
# predicted intentions of the key actors.

# Here's the set of possible actions for the ego vehicle:
# - `accelerate`: Accelerate the ego vehicle
# - `decelerate`: Decelerate the ego vehicle
# - `maintain_speed`: Maintain the ego vehicle's current speed
# - `change_lane_left`: Change the ego vehicle's lane to the left.
# - `change_lane_right`: Change the ego vehicle's lane to the right.

# Your response should be structured as follows:
# plan: A sequence of high-level driving actions for the ego vehicle over a 3s planning horizon. Each action should be
#     separated by a comma.
# reasoning: Brief description justifying the chosen actions for the ego vehicle.
#         """
        system_instruction = f"""
You are a component of an expert autonomous driving assistant, responsible for the planning stack of the pipeline.
You will be given RGB images of the Bird's Eye View (BEV) and front-view with respect to the ego vehicle. Alongside this information,
you will be provided with a textual summary of likely intentions of key actors in the scene generated from the upstream prediction stacks. Given this
information, generate a sequence of high-level driving actions for the ego vehicle over a 3 second planning horizon.

#ENVIRONMENT SETUP#
A 2D BEV coordinate system is used for decision-making and all measurements use the metric system. The setup is as follows:
    1. All positions are given as 2D coordinates in the x-y plane in metres. The x-axis is oriented positive UP and the y-axis is oriented
       positive RIGHT
    2. All orientations are equivalent to the yaw and are given in radians
    3. All distance measurements are given in metres
    4. All speed measurements are given in metres/second

#IMAGE INPUT DETAILS#
In both the front-view and BEV RGB images, all NPC objects are enclosed by their ground-truth bounding boxes and labeled with
their corresponding actor IDs. In the image input, the front-view image is stacked on top of the BEV image. Attend carefully
to critical objects and actors in the front-view image and use the BEV image for additional surrounding context to inform
your decisions. The green line represents the global route the ego vehicle is expected to follow.

#TEXT INPUT DETAILS#
The key actor summary describes key actor intentions as follows:

key_actor_id: ID of the key actor.
key_actor_type: Type of the key actor (e.g., vehicle, pedestrian).
key_actor_intention: Predicted intention of the key actor based on the scene context.
key_actor_reasoning: Step-by-step reasoning justifying the predicted intention of the key actor

#PLANNING TASK#
You are responsible for the following planning task:
1. Generate a sequence of high-level driving actions for the ego vehicle over a 3 second planning horizon. The plan should
prioritize the ego vehicle's safety and conservative decision-making, taking into account the predicted intentions of the key actors.
Each high-level action can be given in increments of 0.5 or 1.0 seconds but the total sequence must add to 3.0 seconds.
Reason step by step to justify why the generated plan is the most optimal.

##EGO VEHICLE AVAILABLE ACTIONS##
The ego vehicle can only execute a set of discrete actions that are listed below:
1. `accelerate`: Accelerate the ego vehicle
2. `decelerate`: Decelerate the ego vehicle
3. `maintain_speed`: Maintain the ego vehicle's current speed
4. `change_lane_left`: Change the ego vehicle's lane to the left.
5. `change_lane_right`: Change the ego vehicle's lane to the right.
6. `stop`: Stop the ego-vehicle

Your response should be structured as follows:

plan: A sequence of high-level driving actions for the ego vehicle over a 3s planning horizon. Each action should be
    separated by a comma.
reasoning: Step-by-step reasoning justifying the chosen actions for the ego vehicle.
        """

        system_message = self.create_system_message(text=system_instruction)
        user_message = self.create_user_message(text=key_actor_intentions, image=image_path)

        messages = [system_message, user_message]

        response = self.send_message(messages)

        return response

In [91]:
scene_analyzer = SceneAnalyzer()

image_path = Path("/home/carla/carla_garage/leaderboard_autopilot/runs/run_20250702_001202/rgb_bounding_boxes_0082.png")
text_path = "/home/carla/carla_garage/leaderboard_autopilot/runs/run_20250702_001202/scene_context_0082.txt"

scene_description = scene_analyzer.interpret_scene(scene_context_path=text_path, image_path=image_path)
print("Scene Description:")
print(scene_description)

actor_intentions = scene_analyzer.predict_intentions(scene_context_path=text_path, image_path=image_path, key_actor_description=scene_description)
print("\nActor Intentions:")
print(actor_intentions)

ego_actions = scene_analyzer.plan_ego_actions(key_actor_intentions=actor_intentions, image_path=image_path)
print("\nEgo Vehicle Actions:")
print(ego_actions)

Scene Description:
road_description: The ego vehicle is traveling on a multi-lane road with clear lane markings and a guardrail on the right side. The road appears to be a local or arterial road with multiple lanes in the same direction.

traffic_description: Traffic is light with one notable vehicle ahead in the right-adjacent lane (Right-1 lane). There are no traffic lights or stop signs reported nearby. The speed limit is approximately 33.3 m/s.

static_objects_and_obstacles_description: There are no static obstacles or objects directly impacting the ego vehicle's path. The right side of the road has a guardrail and some trees.

ego_vehicle_description: The ego vehicle is moving at a speed of 8.65 m/s, oriented almost straight ahead (orientation near zero radians), positioned at coordinates [-513.19, 3104.60]. There is no upcoming lane change planned or possible at this time.

key_actors: The key actor is vehicle ID 3696, located in the right-adjacent lane (Right-1 lane), approximat

In [ ]:
import os
import numpy as np
import base64

from PIL import Image
from openai import OpenAI

class OpenRouterAgent:
    model_names = {
        "openai" : ["openai/gpt-4.1-2025-04-14"],
        "qwen" : ["qwen/qwen2.5-vl-72b-instruct:free"],
        "google" : ["google/gemini-2.5-flash"],
        "anthropic" : ["anthropic/claude-3-7-sonnet-20250219"]
    }

    def __init__(self, model_name: str = "openai/gpt-4.1-2025-04-14"):
        model_provider = model_name.split('/')[0]
        if model_provider not in self.model_names:
            raise ValueError(f"Invalid model provider: {model_provider}. Available providers: {list(self.model_names.keys())}")

        self.model_name = model_name
        if self.model_name not in self.model_names[model_provider]:
            raise ValueError(f"Invalid model name: {self.model_name}. Available models for {model_provider}: {self.model_names[model_provider]}")

        self.api_key = os.getenv("OPENROUTER_API_KEY")
        if not self.api_key:
            raise ValueError("OPENROUTER_API_KEY environment variable is not set.")

        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=self.api_key
        )

class VLMAgent(OpenRouterAgent):
    def __init__(self, temperature: float = 0.1, max_output_tokens: int = 500, model_name: str = "openai/gpt-4.1-2025-04-14"):
        super().__init__(model_name=model_name)

        self.temperature = temperature
        self.max_output_tokens = max_output_tokens

    def __create_message(self, role: str, content: str):
        return {
            "role": role,
            "content": content
        }

    def __create_message_content(self, type: str, text_content: str = None, image_path: str = None):
        def encode_image(image_path: str):
            with open(image_path, "rb") as image_file:
                return base64.b64encode(image_file.read()).decode('utf-8')

        content = {}
        if type == "input_text":
            if not text_content:
                raise ValueError("Text content must be provided for input_text type.")
            content = {
                "type" : type,
                "text" : text_content
            }
        elif type == "input_image":
            if not image_path:
                raise ValueError("Image path must be provided for input_image type.")
            base64_image_data = encode_image(image_path)
            content = {
                "type"          : type,
                "image_url"     : f"data:image/jpeg;base64,{base64_image_data}",
            }
        return content

    def __create_text_message_content(self, text: str):
        text_content = self.__create_message_content(type="input_text", text_content=text)

        return text_content

    def __create_image_message_content(self, image_path: str):
        image_content = self.__create_message_content(type="input_image", image_path=image_path)

        return image_content

    def _create_user_message(self, text: str = None, image_path: str = None):
        user_message_content = []

        if text:
            user_message_content.append(self.__create_text_message_content(text=text))
        if image_path:
            user_message_content.append(self.__create_image_message_content(image_path=image_path))

        user_message = self.__create_message(role="user", content=user_message_content)

        return user_message

    def _create_system_message(self, text: str):
        message = self.__create_message(role="developer", content=text)

        return message

class SceneAnalyzer(VLMAgent):
    def __init__(
            self,
            temperature: float = 0.1,
            max_output_tokens: int = 500,
            model_name: str = "gpt-4.1-2025-04-14"):
        super().__init__(temperature, max_output_tokens, model_name)

    def interpret_scene(self, scene_context_path: str, image_path: str):
        system_instruction = f"""
You are an expert autonomous driving assistant. You will be given RGB images of the Bird's Eye View (BEV) and
front-view with respect to the ego vehicle. Alongside this information, you will be provided with a textual
representation of the scene metadata. Given this information, provide a concise, natural language summary of
the current driving context.

Also, identify relevant actors in the scene based on their pose and proximity to
the ego vehicle. Note that leading vehicles are more important than trailing vehicles in the ego lane, whereas
both are important in adjacent lanes. In the textual summary, lane adjacency is provided as (DIRECTION - n) where
DIRECTION is either "Left" or "Right" and n is the lane number relative to the ego vehicle's lane.

Your response should be structured as follows:

road_description: Description of the road network (junction, highway, local road, etc.).
traffic_description: Description of the traffic conditions in the scene.
static_objects_and_obstacles_description: Description of static objects and obstacles in the scene (if present).
ego_vehicle_description: Description of the state of the ego vehicle.
key_actors: Description of nearby actors with their IDs, poses, and states that are relevant to the driving context.
reasoning: Brief description justifying your choice of key actors.
        """

        with open(scene_context_path, 'r') as file:
            scene_context = file.read()

        system_message = self._create_system_message(text=system_instruction)
        user_message = self._create_user_message(text=scene_context, image_path=image_path)

        messages = [system_message, user_message]

        response = self.client.responses.create(
            model=self.model_name,
            input=messages,
            temperature=self.temperature,
            max_output_tokens=self.max_output_tokens,
        )

        return response.output_text

    def predict_intentions(self, scene_context_path: str, image_path: str, key_actor_description: str):
        system_instruction = f"""
You are an expert autonomous driving assistant. You will be given RGB images of the Bird's Eye View (BEV) and
front-view with respect to the ego vehicle. Alongside this information, you will be provided with a textual
representation of the scene metadata and a corresponding natural language summary. This includes information
about key actors that may be relevant to the ego vehicle's immediate decision-making process.

Given this information, predict the likely intentions of the key actors in the scene considering ego vehicle's
remaining route which is shown as a green line in the image. Opt for predictions that prioritize the ego's safety
and conservative decision-making.

For all actors, consider their current pose, speed, and proximity to the ego vehicle. For vehicles, their set of
possible actions are: "follow_lane", "change_lane_left", "change_lane_right", "turn_left", "turn_right",
"stop", "accelerate", and "decelerate". For pedestrians, their actions are: "cross_street", "wait", and "walk".

Your response should be structured as follows:

key_actor_id: ID of the key actor.
key_actor_type: Type of the key actor (e.g., vehicle, pedestrian).
key_actor_intention: Predicted intention of the key actor based on the scene context.
key_actor_reasoning: Brief description justifying the predicted intention of the key actor.
        """

        with open(scene_context_path, 'r') as file:
            scene_context = file.read()

        system_message = self._create_system_message(text=system_instruction)

        separator_text = "\n\nKey Actor Description:\n"
        scene_context_with_key_actor = scene_context + separator_text + key_actor_description
        user_message = self._create_user_message(text=scene_context_with_key_actor, image_path=image_path)

        messages = [system_message, user_message]

        response = self.client.responses.create(
            model=self.model_name,
            input=messages,
            temperature=self.temperature,
            max_output_tokens=self.max_output_tokens,
        )

        return response.output_text

    def plan_ego_actions(self, key_actor_intentions: str, image_path: str):
        system_instruction = f"""
You are an expert autonomous driving assistant. You will be given RGB images of the Bird's Eye View (BEV) and
front-view with respect to the ego vehicle. Alongside this information, you will be provided with a textual
summary of likely intentions of key actors in the scene.

Given this information, generate a sequence of high-level driving actions for the ego vehicle over a 3s planning
horizon. The plan should prioritize the ego vehicle's safety and conservative decision-making, taking into account the
predicted intentions of the key actors.

Here's the set of possible actions for the ego vehicle:
- `accelerate`: Accelerate the ego vehicle
- `decelerate`: Decelerate the ego vehicle
- `maintain_speed`: Maintain the ego vehicle's current speed
- `change_lane_left`: Change the ego vehicle's lane to the left.
- `change_lane_right`: Change the ego vehicle's lane to the right.

Your response should be structured as follows:
plan: A sequence of high-level driving actions for the ego vehicle over a 3s planning horizon. Each action should be
    separated by a comma.
reasoning: Brief description justifying the chosen actions for the ego vehicle.
        """

        system_message = self._create_system_message(text=system_instruction)
        user_message = self._create_user_message(text=key_actor_intentions, image_path=image_path)

        messages = [system_message, user_message]

        response = self.client.responses.create(
            model=self.model_name,
            input=messages,
            temperature=self.temperature,
            max_output_tokens=self.max_output_tokens,
        )

        return response.output_text

In [23]:
scene_analyzer = SceneAnalyzer()

image_path = "/home/carla/carla_garage/leaderboard_autopilot/runs/run_20250702_001202/rgb_bounding_boxes_0079.png"
text_path = "/home/carla/carla_garage/leaderboard_autopilot/runs/run_20250702_001202/scene_context_0079.txt"

scene_description = scene_analyzer.interpret_scene(scene_context_path=text_path, image_path=image_path)
print("Scene Description:")
print(scene_description)

actor_intentions = scene_analyzer.predict_intentions(scene_context_path=text_path, image_path=image_path, key_actor_description=scene_description)
print("\nActor Intentions:")
print(actor_intentions)

ego_actions = scene_analyzer.plan_ego_actions(key_actor_intentions=actor_intentions, image_path=image_path)
print("\nEgo Vehicle Actions:")
print(ego_actions)

TypeError: Image must be a numpy array or a Path object pointing to an image file.